In [ ]:
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd

# Load dataset
dataset = load_dataset("0xnbk/resume-ats-score-v1-en")
train_df = pd.DataFrame(dataset['train'])

# Prepare training examples with normalized scores (0-1 range)
train_examples = []
for _, row in train_df.iterrows():
    # Split resume and job description
    resume, job = row['text'].split(' SEP ')
    # Normalize score to 0-1 range for CosineSimilarityLoss
    normalized_score = row['ats_score'] / 100.0
    train_examples.append(
        InputExample(texts=[resume, job], label=normalized_score)
    )

# Load base model
model = SentenceTransformer('jinaai/jina-embeddings-v2-base-en')

# Define loss and dataloader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

# Train
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    optimizer_params={'lr': 2e-5},
    output_path='./ats-semantic-model'
)

# Save
model.save('./ats-semantic-model')